## DATASET PREPROCESSING

Tools yang digunakan:
Data mentah dikumpulkan menggunakan Instagram Scraper dari platform Apify (apify/instagram-scraper), yang memungkinkan pencarian dan ekstraksi data profil Instagram secara otomatis berdasarkan kata kunci maupun hashtag tertentu.

Metode pencarian:
Pengumpulan data dilakukan melalui dua pendekatan:

Pencarian berbasis keyword search. menggunakan fitur "Search by query" pada Instagram Scraper, yang menelusuri profil publik Instagram melalui data pencarian Google dan Facebook Ads Library.
Pencarian berbasis hashtag. menelusuri post-post yang menggunakan tagar tertentu untuk menemukan akun-akun terkait.

Kata kunci yang digunakan:
Restoran Sehat
Healthy food
Makanan diet sehat
Gizi seimbang
Salad bar Indonesia
Menurunkan kolesterol
Bebas gula tanpa pemanis

### Import Dataset and combine all (the dataset is separated by the keyword that being used)

In [11]:
import pandas as pd
import glob
import os

folder_path = '/Users/annya/scrapping_instagram/Raw_Dataset'
csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

print(f"Ditemukan {len(csv_files)} file CSV:")
for f in csv_files:
    print(f" - {f}")

dfs = []
for filepath in csv_files:
    df = pd.read_csv(filepath)
    df['source_file'] = os.path.basename(filepath)
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
print(f"Total baris setelah digabung: {len(combined_df)}")

Ditemukan 9 file CSV:
 - /Users/annya/scrapping_instagram/Raw_Dataset/dataset_instagram-scraper_2026-09-17_03-04-04-919.csv
 - /Users/annya/scrapping_instagram/Raw_Dataset/dataset_instagram-scraper_2026-09-17_03-06-32-440 (1).csv
 - /Users/annya/scrapping_instagram/Raw_Dataset/dataset_instagram-scraper_2026-09-17_02-57-02-251.csv
 - /Users/annya/scrapping_instagram/Raw_Dataset/dataset_instagram-scraper_2026-09-17_02-56-06-387.csv
 - /Users/annya/scrapping_instagram/Raw_Dataset/dataset_instagram-scraper_2026-09-17_02-51-33-223.csv
 - /Users/annya/scrapping_instagram/Raw_Dataset/dataset_instagram-scraper_2026-09-17_03-06-32-440.csv
 - /Users/annya/scrapping_instagram/Raw_Dataset/dataset_instagram-scraper_2026-09-17_03-06-33-424.csv
 - /Users/annya/scrapping_instagram/Raw_Dataset/dataset_instagram-scraper_2026-09-17_02-49-47-128.csv
 - /Users/annya/scrapping_instagram/Raw_Dataset/dataset_instagram-scraper_2026-09-17_02-37-22-191.csv
Total baris setelah digabung: 177


### Drop Baris Kosong

In [12]:
combined_df = combined_df.dropna(how='all')
print(f"Total baris setelah buang baris kosong total: {len(combined_df)}")

Total baris setelah buang baris kosong total: 177


### Drop Baris Duplicate (berdasarkan biography)

In [13]:
is_bio_kosong = combined_df['biography'].isna() | (combined_df['biography'].astype(str).str.strip() == '')

bio_kosong = combined_df[is_bio_kosong]        
bio_ada_isi = combined_df[~is_bio_kosong]      

print(f"Baris dengan biography kosong (disimpan semua): {len(bio_kosong)}")
print(f"Baris dengan biography terisi (akan di-dedupe): {len(bio_ada_isi)}")

jumlah_duplikat = bio_ada_isi.duplicated(subset=['biography']).sum()
print(f"Jumlah duplikat ditemukan (by biography): {jumlah_duplikat}")

bio_ada_isi = bio_ada_isi.drop_duplicates(subset=['biography'], keep='first')

combined_df = pd.concat([bio_ada_isi, bio_kosong], ignore_index=True)

print(f"Total baris akhir: {len(combined_df)}")


Baris dengan biography kosong (disimpan semua): 9
Baris dengan biography terisi (akan di-dedupe): 168
Jumlah duplikat ditemukan (by biography): 3
Total baris akhir: 174


### Drop Baris dengan status private dan yang followers 0

In [14]:
combined_df = combined_df[
    (combined_df['private'] != True) & 
    (combined_df['followersCount'] != 0)
]
print(f"Setelah drop private/followers 0: {len(combined_df)}")

Setelah drop private/followers 0: 171


### Drop Kolom tidak Relevan

In [15]:
kolom_penting = [
    'username', 'fullName', 'biography', 'followersCount', 'followsCount',
    'postsCount', 'externalUrl', 'businessCategoryName', 'verified',
    'private', 'isBusinessAccount', 'searchTerm', 'searchSource', 'source_file'
]
kolom_tersedia = [k for k in kolom_penting if k in combined_df.columns]
combined_df = combined_df[kolom_tersedia]
print(f"Kolom akhir: {combined_df.columns.tolist()}")

Kolom akhir: ['username', 'fullName', 'biography', 'followersCount', 'followsCount', 'postsCount', 'externalUrl', 'businessCategoryName', 'verified', 'private', 'isBusinessAccount', 'searchTerm', 'searchSource', 'source_file']


### Memastikan bahwa akun berasal dari Indonesia (memakai language detector dan juga keyword indonesia --otomatis maupun manual)

In [16]:
from langdetect import detect, LangDetectException
import re

def detect_lang_safe(text):
    """Deteksi bahasa dengan aman, return None kalau gagal/teks terlalu pendek"""
    if pd.isna(text) or str(text).strip() == '':
        return None
    try:
        return detect(str(text))
    except LangDetectException:
        return None

print("Mendeteksi bahasa... (bisa agak lama kalau datanya banyak)")
combined_df['detected_lang'] = combined_df['biography'].apply(detect_lang_safe)

print(combined_df['detected_lang'].value_counts())

mask_keep = combined_df['detected_lang'].isin(['id', 'en']) | combined_df['detected_lang'].isna()

print(f"Total baris sebelum filter bahasa: {len(combined_df)}")
combined_df = combined_df[mask_keep]
print(f"Total baris setelah filter bahasa: {len(combined_df)}")

combined_df = combined_df.drop(columns=['detected_lang'])

Mendeteksi bahasa... (bisa agak lama kalau datanya banyak)
detected_lang
en    84
id    20
vi    11
es     7
pt     5
ru     4
it     4
de     3
fr     3
af     2
no     2
fa     2
pl     2
th     2
bg     1
tl     1
uk     1
tr     1
ar     1
ko     1
so     1
Name: count, dtype: int64
Total baris sebelum filter bahasa: 171
Total baris setelah filter bahasa: 117


In [17]:
kota_indo = ['jakarta','bandung','surabaya','medan','semarang','makassar',
    'palembang','depok','tangerang','bekasi','bogor','yogyakarta','jogja',
    'malang','denpasar','bali','solo','surakarta','balikpapan','pekanbaru',
    'batam','padang','manado','samarinda','pontianak','jambi','cirebon',
    'sukabumi','tasikmalaya','cimahi','pematangsiantar','sampit',
    'banjarmasin','mataram','kupang','ambon','jayapura','bengkulu','palu',
    'kendari','gorontalo']

kata_indo = ['jalan','jl\\.','kelurahan','kecamatan','kabupaten','ruko','gang',
    'rt/rw','wa only','chat wa','order via','pemesanan','halal','sudah bpom',
    'terima kasih','buka setiap hari','senin-minggu','sehat','gizi','diet',
    'resep','toko','produk','pesan','khusus','dapur','katering','masakan',
    'ongkir','bergizi']

bank_kurir_indo = ['bca','bri','mandiri','bni','jnt','jne','sicepat','tiki','j&t']

def cek_sinyal_indo(row):
    teks = f"{row['biography']} {row['fullName']}".lower()
    cek_nomor = bool(re.search(r'(\+62|08)\d{8,12}', teks.replace(' ', '').replace('-', '')))
    cek_kota = any(kota in teks for kota in kota_indo)
    cek_kata = any(re.search(kata, teks) for kata in kata_indo)
    cek_bank = any(bank in teks for bank in bank_kurir_indo)
    return cek_nomor or cek_kota or cek_kata or cek_bank

combined_df['sinyal_indo'] = combined_df.apply(cek_sinyal_indo, axis=1)

print(f"\nTotal sebelum filter Indonesia: {len(combined_df)}")
combined_df = combined_df[combined_df['sinyal_indo']].drop(columns=['sinyal_indo'])
print(f"Total setelah filter Indonesia: {len(combined_df)}")


Total sebelum filter Indonesia: 117
Total setelah filter Indonesia: 40


### Memastikan Dataset rapi dan menyimpan dataset ke Preprocessing_Data

In [18]:
combined_df = combined_df.sort_values('followersCount', ascending=False).reset_index(drop=True)

output_path = os.path.join('/Users/annya/scrapping_instagram/Preprocessing_Data', 'combined_final.csv')
combined_df.to_csv(output_path, index=False)
print(f"✅ Disimpan ke: {output_path}")
print(f"Total baris final: {len(combined_df)}")

✅ Disimpan ke: /Users/annya/scrapping_instagram/Preprocessing_Data/combined_final.csv
Total baris final: 40
